# Phân loại toàn bộ danh mục Thủ Đức qua Vilao — draft

Chủ dự án yêu cầu chạy toàn bộ 219 món. Notebook dùng chat `vgpt/gpt-5.6-luna`, cùng codebook thử nghiệm; giữ nguyên CSV nguồn và thêm cột taxonomy vào file riêng. Không publish seed hoặc thay trạng thái GM-03. Model chỉ nhận tên/mô tả/nhóm menu công khai, không nhận địa chỉ, GPS, ảnh, giá hay dữ liệu người dùng.

**Cache theo từng món + journal từng batch.** Tái dùng 9 dòng pilot đạt contract với provenance riêng; dòng pilot lỗi được gọi lại. Tối đa 10 món/request, 2 attempt/món, 44 request/lượt; khi HTTP lỗi dừng và giữ checkpoint. Dòng schema lỗi có tối đa một lần thử lại, không sửa nhãn tự động. Số token provider báo được theo dõi với ngưỡng dừng 200.000 cho batch mới; đây là ngưỡng kiểm sau response, chưa là trần tiền/token từ provider. `max_tokens=1400` đã có quan sát completion gồm reasoning vượt ngưỡng gửi.

Run All mặc định dùng cache, không gọi API. Đặt `ALLOW_API_CALLS=True` ở cell đầu hoặc runner `--allow-api` để xử lý phần thiếu. Đã xong thì gọi lại vẫn không tốn request. Không dùng `--refresh` cho notebook này; thay prompt/codebook tạo namespace cache mới có chủ đích.


## 1. Đường dẫn, runtime và giới hạn

Chỉ dùng thư viện có sẵn. Thư mục raw/processed/reports được tự tạo; source `.env` giữ riêng, không có key trong notebook hoặc báo cáo. Kernel đặt ngoài repo thì điền DATA_PREPARATION_DIR.


In [ ]:
from pathlib import Path
import sys, json, platform
import pandas as pd
DATA_PREPARATION_DIR = None
ALLOW_API_CALLS = False
MAX_HTTP_CALLS = 44
MAX_ITEM_ATTEMPTS = 2
MAX_REPORTED_TOKENS = 200000
options = globals().get('NOTEBOOK_OPTIONS', {})
ALLOW_API_CALLS = (ALLOW_API_CALLS or bool(options.get('allow_api'))) and not options.get('offline',False)
if options.get('refresh'):
    raise ValueError('Không refresh cache full; đổi prompt/config có chủ đích để tạo namespace mới.')
BASE = Path(DATA_PREPARATION_DIR).resolve() if DATA_PREPARATION_DIR else next((
    p if p.name=='data-preparation' else p/'data-preparation'
    for p in (Path.cwd(),*Path.cwd().parents)
    if (p if p.name=='data-preparation' else p/'data-preparation').joinpath('config/taxonomy_pilot_v1.json').is_file()),None)
if BASE is None: raise ValueError('Đặt DATA_PREPARATION_DIR để tìm project.')
BASE=BASE.resolve()
sys.path.insert(0,str(BASE/'scripts'))
from vilao_pilot import Config,PilotClient,load_config,compact,sha
from vilao_catalogue import CatalogueRun,PROMPT_VERSION,atomic_json
SOURCE=BASE/'datasets/thu-duc-menus/processed/thu_duc_dishes_final.csv'
taxonomy=json.loads((BASE/'config/taxonomy_pilot_v1.json').read_text('utf-8'))
client=PilotClient(load_config(BASE.parent/'.env') if ALLOW_API_CALLS else Config('offline-placeholder'),max_calls=MAX_HTTP_CALLS)
run=CatalogueRun(BASE,client,SOURCE,taxonomy,max_attempts=MAX_ITEM_ATTEMPTS,max_reported_tokens=MAX_REPORTED_TOKENS)
REPORTS_DIR=run.reports
display({'python':platform.python_version(),'pandas':pd.__version__,'model':client.config.model,
         'allow_api':ALLOW_API_CALLS,'max_http_calls':MAX_HTTP_CALLS,'attempts_per_item':MAX_ITEM_ATTEMPTS,
         'max_reported_token_threshold':MAX_REPORTED_TOKENS,'dataset':str(run.root),'cache_namespace':run.scope})


## 2. Xem nguồn và context từng món

219 dòng, 35 cột nguồn; các ID phải duy nhất. Một offering ví dụ cung cấp tên/mô tả/nhóm cùng nguồn. Alias/variant nguồn giữ nguyên. Không dùng địa chỉ bán làm origin. Không tự cắt bỏ mô tả quan trọng khi vượt giới hạn byte.


In [ ]:
source_df=pd.DataFrame(run.originals)
sample_df=pd.DataFrame(run.samples)
display({'rows':len(source_df),'columns':len(source_df.columns),'source_bytes':SOURCE.stat().st_size,
         'source_sha256':run.source_hash,'missing_descriptions':sum(not s['description'] for s in run.samples),
         'unique_dish_ids':source_df.dish_id.nunique()})
display(sample_df[['dish_name','menu_name','description','groups']].head(20))
display(source_df.groupby('category').size().to_frame('rows'))


## 3. Tái dùng pilot và xem phần cần chạy

Chỉ nhận pilot có model/codebook/context trùng, validator đạt. Cache item mới gồm hash context (có ID), model/prompt/contract, batch nguồn và status. Batch journal được đọc lại để phục hồi nếu response đã lưu nhưng item cache chưa ghi hết. Nhãn hợp lệ đã cache không bị gọi hoặc ghi đè.


In [ ]:
imported=run.import_pilot()
pending=run.pending()
display({'new_pilot_imports':imported,'cached_valid':sum(s['status']=='schema_valid_draft' for s in run.states.values()),
         'remaining_attemptable':len(pending),'initial_batches_at_most':(len(pending)+9)//10,
         'hard_request_limit_this_run':MAX_HTTP_CALLS})
display(pd.DataFrame([{'dish_name':s['dish_name'],'status':run.states.get(s['dish_id'],{}).get('status','pending'),
                      'attempts':len(run.states.get(s['dish_id'],{}).get('attempts',[]))} for s in run.samples]).head(25))


## 4. Codebook và prompt batch

Array 9 vị trí: index, cuisines, origin, categories, methods, temperature, 4 mức vị, meal_slots và 7 evidence. Chỉ nhận mã thuộc codebook. Sửa prompt so với pilot: nhắc lại unknown phải có evidence 0, kể cả nguồn nói rõ có lựa chọn cay/không cay. Cuisine/category/method được suy luận và ghi evidence 2; nhiệt độ/vị/buổi/vùng Việt cần menu nêu. Chuẩn cấu trúc không chứng minh nhãn thực tế đúng.


In [ ]:
for group in ('cuisines','origin','categories','methods','temperature','flavor_levels','meal_slots','evidence'):
    display(pd.DataFrame([{'field':group,'code':int(k),'label':v} for k,v in taxonomy[group].items()]))
print(run.system)
preview=run.messages((pending or run.samples)[:10])
print(preview[1]['content'])
display({'prompt_version':PROMPT_VERSION,'request_bytes':len(compact({'model':client.config.model,'messages':preview,'max_tokens':1400,'stream':False}).encode()),
         'max_request_bytes':16000,'max_tokens_requested':1400})


## 5. Chạy hoặc tiếp tục từ cache, checkpoint sau từng batch

Ghi journal in_flight trước HTTP; response hoàn chỉnh lưu atomic trước item cache. Dừng khi HTTP lỗi, vượt giới hạn request hoặc đạt ngưỡng usage; output checkpoint vẫn có đủ dòng nguồn với trạng thái pending/needs_review. Request bị ngắt trước khi lưu response được ghi uncertain: không tự gửi trùng, cần kiểm journal trước. Server không có chứng nhận exactly-once, nên không khẳng định request mất response chưa bị tính phí.

Schema sai chỉ thử lại một lần; cả phản hồi cũ và mới vẫn giữ. Dòng chưa đạt không có nhãn được chấp nhận trong cột taxonomy, nhưng có raw array và lỗi để rà. CSV schema_valid_draft vẫn cần human review.

Parser có thể thêm/bỏ đúng một dấu ] cuối wrapper JSON nếu số dòng/shape/index vẫn đúng; raw response giữ nguyên, không sửa giá trị nhãn. Cache ghi json_wrapper_repair; mỗi dòng vẫn qua validator.


In [ ]:
def progress(message):
    print(message,flush=True)
    if options and sys.__stdout__ is not None:
        sys.__stdout__.write(message+'\n');sys.__stdout__.flush()
stats=run.execute(allow_api=ALLOW_API_CALLS,progress=progress)
display(stats)
if not stats['complete']:
    print('Chưa đủ dòng đạt contract; xem needs_review.json. API bật sẽ chỉ chạy phần còn thiếu/chưa hết attempt.')


## 6. Kiểm CSV cuối, dữ liệu gốc và nhãn

UTF-8 có BOM, 219 dòng duy nhất, 35 cột gốc bảo toàn từng ô; thêm 17 cột taxonomy. Các cột gốc cuisine/origin/temperature/flavor/meal_slots không bị sửa: dùng taxonomy_* để xem đề xuất mới. Mỗi dòng giữ nguyên alias/variant/ảnh/mô tả/giá/URL/metadata đầy đủ của nguồn. Unknown khác “không cay” hoặc nhiệt độ thường.


In [ ]:
FINAL=run.processed/'thu_duc_dishes_taxonomy.csv'
final_df=pd.read_csv(FINAL,encoding='utf-8-sig',dtype=str,keep_default_na=False)
assert final_df.dish_id.is_unique and len(final_df)==len(source_df)
assert final_df[run.columns].to_dict('records')==run.originals
assert FINAL.read_bytes().startswith(b'\xef\xbb\xbf')
assert sha(SOURCE.read_bytes())==run.source_hash
display(final_df[['dish_name','taxonomy_cuisines_json','taxonomy_categories_json','taxonomy_methods_json',
                  'taxonomy_origin','taxonomy_temperature','taxonomy_flavors_json','taxonomy_meal_slots_json','taxonomy_status']].head(30))
display(final_df.groupby('taxonomy_status').size().to_frame('rows'))
issues=json.loads((REPORTS_DIR/'needs_review.json').read_text('utf-8'))
display(pd.DataFrame(issues))


## 7. Báo cáo và hướng dẫn đọc cột

run_report giữ usage của các batch mới một lần duy nhất, không cộng lại khi replay cache; 9 pilot reuse được ghi riêng. CSV có link batch cache, hash scope và model/timestamp. Báo cáo không chứng nhận độ chính xác, lịch quán/tồn món, thành phần/dị ứng hoặc quyền ảnh. Notebook được kiểm bằng Python exec, chưa kiểm trực tiếp Jupyter kernel.


In [ ]:
lines=['# Danh mục Thủ Đức có taxonomy đề xuất','',
       'Source: 219 món từ menu khảo sát Thủ Đức cũ; CSV nguồn giữ nguyên.',
       'Model: `'+client.config.model+'`; prompt '+PROMPT_VERSION+'.',
       'Đầu ra: '+str(stats['output_rows'])+' dòng, '+str(stats['output_columns'])+' cột; '+str(stats['schema_valid_rows'])+' dòng đạt contract, '+str(stats['needs_review_rows'])+' dòng chưa đạt.',
       'Tái dùng pilot: '+str(stats['pilot_reused_rows'])+' dòng. Batch mới đã lưu: '+str(stats['cached_new_batch_requests'])+'.',
       'Usage batch mới do provider báo: `'+compact(stats['usage_new_batches'])+'`; không cộng lại khi đọc cache.',
       'Batch có completion_tokens vượt max_tokens đã gửi: '+str(stats['completion_limit_exceeded_batches'])+'. Chưa có trần tiền được kiểm chứng.',
       'Dòng phục hồi một dấu ngoặc cuối wrapper JSON, giữ nguyên nhãn/raw: '+str(stats.get('json_wrapper_repair_rows',0))+'.',
       '', '## Các cột bổ sung','',
       '| Cột | Ý nghĩa |','|---|---|',
       '| taxonomy_cuisines_json | Cuisine đề xuất, array nhãn |',
       '| taxonomy_origin | Bắc/Trung/Nam khi nguồn nêu; unknown/not_applicable riêng |',
       '| taxonomy_categories_json | Họ món, tối đa 3 |',
       '| taxonomy_methods_json | Cách nấu món/thành phần chính, tối đa 3 |',
       '| taxonomy_temperature | Nhiệt độ phục vụ theo menu; không suy từ cay |',
       '| taxonomy_flavors_json | Cay/mặn/ngọt/chua: unknown/không có/nhẹ/vừa/nhiều/có chưa rõ mức |',
       '| taxonomy_meal_slots_json | Slot theo menu/giờ nêu, chưa là lịch bán đã xác minh |',
       '| taxonomy_evidence_json | model_inferred_draft/menu_explicit/unknown cho từng nhóm |',
       '| taxonomy_status | schema_valid_draft/pending/needs_review; chưa human approved |',
       '| taxonomy_errors_json | Lỗi contract/response từng dòng |',
       '| taxonomy_raw_row_json | Array nguyên trạng trả về (index trong batch) |',
       '| taxonomy_cache_key | Hash context item |',
       '| taxonomy_prompt_version | Prompt v2 hoặc pilot-v1-reused |',
       '| taxonomy_model, taxonomy_response_model | Model yêu cầu và model provider báo |',
       '| taxonomy_checked_at, taxonomy_batch_reference | Timestamp và đường cache nguồn |',
       '', '35 cột gốc giữ nguyên, kể cả unknown cũ. Nhãn mới nằm trong taxonomy_*.',
       'Nhãn inference và menu_explicit đều cần người rà; schema pass không phải độ chính xác.',
       'Run All mặc định dùng cache, không gọi API. Bật ALLOW_API_CALLS để chạy phần chưa có. Không append hay đếm usage hai lần.',
       'Dừng HTTP lỗi giữ CSV checkpoint đủ source rows; phần chưa có nhãn ghi pending. needs_review.json liệt kê toàn bộ dòng chưa đạt.']
(REPORTS_DIR/'DATASET_REPORT.md').write_text('\n'.join(lines)+'\n','utf-8')
display({'final_csv':str(FINAL),'report':str(REPORTS_DIR/'DATASET_REPORT.md'),'run_report':str(REPORTS_DIR/'run_report.json')})

quality = {'unknown_fields': {k: sum(r[k] in ('unknown','["unknown"]') for r in final_df.to_dict('records'))
                             for k in ('taxonomy_cuisines_json','taxonomy_origin','taxonomy_temperature','taxonomy_meal_slots_json')},
           'source_category_not_in_proposal': [
               {'dish_id':r['dish_id'],'dish_name':r['dish_name'],'original_category':r['category'],
                'proposed_categories':json.loads(r['taxonomy_categories_json'])}
               for r in final_df.to_dict('records') if r['taxonomy_categories_json'] and r['category'] not in json.loads(r['taxonomy_categories_json'])],
           'note':'Source category and model labels both draft; matching is not ground-truth accuracy.'}
quality['unknown_fields']['taxonomy_flavors_all_unknown'] = sum(
    bool(r['taxonomy_flavors_json']) and all(v=='unknown' for v in json.loads(r['taxonomy_flavors_json']).values())
    for r in final_df.to_dict('records'))
atomic_json(REPORTS_DIR/'quality_summary.json',quality)
display(quality)
